# Frame Scene/Event Detection Extractor v1

Đọc output vector embedding từ GCS, gom keyframe liên tiếp thành event bằng time gap + cosine similarity, rồi upload `events`, `map-event`, `events.jsonl` và summary lên GCS.


## 1. Parameters

**Note:** Đây là cell chính cần chỉnh trên Kaggle. Notebook này giả định `fe-vector-embedding-v1.ipynb` đã chạy thành công trước đó.


In [ ]:
from types import SimpleNamespace

# GCS and dataset identity. Empty bucket reads Kaggle Secret GCS_BUCKET.
GCS_BUCKET = "aic_ai_2026"
GCS_BUCKET_SECRET_NAME = "GCS_BUCKET"
GCS_CREDENTIALS_FILE = ""
GCS_CREDENTIALS_JSON_SECRET_NAME = "GCS_CREDENTIALS_JSON"

DATASET_ID = "ai_challenge_2025"
PROFILE_VERSION = "autoshot_v1"
BATCHES = ["L21"]

# Frame extraction layout on GCS.
KEYFRAMES_PREFIX = "processed/keyframes"
MANIFESTS_PREFIX = "processed/keyframes_manifests"
INPUT_MANIFEST_URI = ""  # Optional explicit gs://.../shot_segments.csv or frames_manifest.jsonl.

# Extractor output layout on GCS.
OUTPUT_PREFIX = "features/extractors"
EXTRACTOR_VERSION = "fe-scene-detection-v1"
ANNOTATION_VERSION = "fe-scene-detection-v1"

# Kaggle local runtime.
RUN_ROOT = "/kaggle/working/feature_extractor_runs"
SCRATCH_DIR = "/kaggle/working/feature_extractor_scratch"
CLEANUP_LOCAL_FRAMES_AFTER_RUN = True

# Execution controls.
DRY_RUN_MAX_FRAMES = 20
DEMO_BATCHES = ["L21"]
DEMO_MAX_FRAMES = 64
FULL_MAX_FRAMES = None
CONFIRM_FULL_RUN = ""  # Set to RUN_FULL_DATASET before full run.

# Resume and failure behavior.
UPLOAD_TO_GCS = True
UPLOAD_RUN_ARTIFACTS = True
SKIP_EXISTING = True
OVERWRITE = False
RESUME_ANNOTATIONS_URI = ""  # Optional gs://.../annotations.jsonl used when SKIP_EXISTING=True.
FAIL_FAST = False

# Parallelism/progress. Tune these for Kaggle GPU/CPU size.
DOWNLOAD_WORKERS = 8
UPLOAD_WORKERS = 8
PIPELINE_BATCH_SIZE = 64
LOG_EVERY_N_FRAMES = 128
USE_TQDM = True

# Task-specific settings are below.
VECTOR_RUN_PREFIX = ""  # Optional gs://.../run_id=... prefix. Empty = latest successful vector run for each batch.
MAX_TIME_GAP_SEC = 6.0
SCENE_SIMILARITY_THRESHOLD = 0.72
MAX_EVENT_DURATION_SEC = 45.0
SEGMENTATION_VERSION = "fe-scene-detection-v1"

cfg = SimpleNamespace(**{name: value for name, value in globals().copy().items() if name.isupper() and not name.startswith("_")})
print("Parameters loaded for Frame Scene/Event Detection Extractor v1.")
print("Batches:", cfg.BATCHES, "Demo:", cfg.DEMO_BATCHES, "Upload:", cfg.UPLOAD_TO_GCS)

EXTRACTOR_NAME = "scene_detection"
cfg.EXTRACTOR_NAME = EXTRACTOR_NAME


## 2. Install Dependencies

**Note:** Chạy cell cài đặt một lần sau khi mở Kaggle session. Nếu Kaggle tải package/model từ Internet, bật Internet trong Notebook Settings.


In [ ]:
%pip install -q google-cloud-storage pandas tqdm numpy


## 3. Shared GCS, Manifest, Run Helpers

**Note:** Cell này chứa helper chung để đọc manifest từ GCS, tải frame, ghi artifact và upload kết quả. Nếu sửa helper, chạy lại cell này trước khi chạy dry/demo/full.


In [ ]:
from __future__ import annotations

import csv
import json
import logging
import os
import shutil
import time
import uuid
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


@dataclass
class RunLayout:
    """Local and GCS paths for one extractor run."""
    run_id: str
    run_dir: Path
    frames_dir: Path
    artifacts_dir: Path
    output_prefix: str
    annotations_path: Path
    errors_path: Path
    metrics_path: Path
    summary_path: Path
    log_path: Path


def utc_now() -> str:
    """Return an ISO-8601 UTC timestamp."""
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def cfg_value(config: Any, name: str, default: Any = None) -> Any:
    """Read a value from a SimpleNamespace-like config object."""
    return getattr(config, name, default)


def normalize_prefix(value: str) -> str:
    """Normalize a GCS object prefix without leading/trailing slashes."""
    return str(value or "").strip().strip("/")


def make_run_id(kind: str) -> str:
    """Create a unique, sortable run identifier."""
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{kind}_{stamp}_{uuid.uuid4().hex[:8]}"


def read_kaggle_secret(secret_name: str) -> str:
    """Read a Kaggle secret if the notebook is running on Kaggle."""
    if not secret_name:
        return ""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name) or ""
    except Exception:
        return ""


def resolve_bucket_name(config: Any, require: bool = True) -> str:
    """Resolve the target GCS bucket from params, env vars, or Kaggle Secrets."""
    configured = str(cfg_value(config, "GCS_BUCKET", "") or "").strip()
    env_value = os.environ.get("GCS_BUCKET", "").strip()
    secret_name = str(cfg_value(config, "GCS_BUCKET_SECRET_NAME", "GCS_BUCKET") or "").strip()
    resolved = configured or env_value or read_kaggle_secret(secret_name).strip()
    if resolved.startswith("gs://"):
        resolved = resolved[len("gs://"):].split("/", 1)[0]
    if require and not resolved:
        raise RuntimeError("Set GCS_BUCKET in params, env vars, or Kaggle Secrets.")
    return resolved


def make_storage_client(config: Any):
    """Create a google-cloud-storage client using Kaggle Secrets or ADC."""
    from google.cloud import storage

    credentials_file = str(cfg_value(config, "GCS_CREDENTIALS_FILE", "") or os.environ.get("GCS_CREDENTIALS_FILE", "")).strip()
    credentials_json = os.environ.get("GCS_CREDENTIALS_JSON", "").strip()
    secret_name = str(cfg_value(config, "GCS_CREDENTIALS_JSON_SECRET_NAME", "GCS_CREDENTIALS_JSON") or "").strip()
    credentials_json = credentials_json or read_kaggle_secret(secret_name).strip()
    if credentials_json:
        from google.oauth2 import service_account
        credentials = service_account.Credentials.from_service_account_info(json.loads(credentials_json))
        return storage.Client(project=credentials.project_id, credentials=credentials)
    if credentials_file:
        return storage.Client.from_service_account_json(credentials_file)
    return storage.Client()


def parse_gcs_uri(uri: str) -> tuple[str, str]:
    """Parse gs://bucket/object into bucket and object name."""
    if not str(uri).startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got {uri}")
    bucket, _, blob = str(uri)[5:].partition("/")
    if not bucket or not blob:
        raise ValueError(f"Invalid GCS URI: {uri}")
    return bucket, blob


def make_output_prefix(config: Any, batch_id: str, run_id: str) -> str:
    """Build the task output prefix for one logical batch/run."""
    return (
        f"{normalize_prefix(cfg_value(config, 'OUTPUT_PREFIX', 'features/extractors'))}/"
        f"dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/"
        f"frame_profile={cfg_value(config, 'PROFILE_VERSION')}/"
        f"extractor={cfg_value(config, 'EXTRACTOR_NAME')}/"
        f"extractor_version={cfg_value(config, 'EXTRACTOR_VERSION')}/"
        f"run_id={run_id}/"
    )


def make_run_layout(config: Any, batch_id: str, run_kind: str) -> RunLayout:
    """Create local directories and output files for a run."""
    run_id = make_run_id(run_kind)
    run_dir = Path(str(cfg_value(config, "RUN_ROOT", "/kaggle/working/feature_extractor_runs"))) / run_id
    frames_dir = run_dir / "frames"
    artifacts_dir = run_dir / "artifacts"
    frames_dir.mkdir(parents=True, exist_ok=True)
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    return RunLayout(run_id, run_dir, frames_dir, artifacts_dir, make_output_prefix(config, batch_id, run_id), artifacts_dir / "annotations.jsonl", artifacts_dir / "errors.jsonl", artifacts_dir / "metrics.csv", artifacts_dir / "summary.json", run_dir / "run.log")


def setup_logging(layout: RunLayout, verbose: bool = False) -> logging.Logger:
    """Configure console and file logging for a notebook run."""
    logger = logging.getLogger(str(cfg_value(cfg, "EXTRACTOR_NAME", "feature_extractor")))
    logger.setLevel(logging.DEBUG if verbose else logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    stream = logging.StreamHandler()
    stream.setFormatter(fmt)
    file_handler = logging.FileHandler(layout.log_path, encoding="utf-8")
    file_handler.setFormatter(fmt)
    logger.addHandler(stream)
    logger.addHandler(file_handler)
    return logger


def upload_file(bucket: Any, local_path: Path, object_key: str, content_type: str = "application/octet-stream") -> None:
    """Upload one local file to GCS."""
    bucket.blob(object_key).upload_from_filename(str(local_path), content_type=content_type, timeout=900)


def upload_text(bucket: Any, text: str, object_key: str, content_type: str = "text/plain") -> None:
    """Upload text content to GCS."""
    bucket.blob(object_key).upload_from_string(text, content_type=content_type, timeout=300)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    """Write a JSON object to disk with UTF-8 encoding."""
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def append_jsonl(path: Path, records: Iterable[dict[str, Any]]) -> int:
    """Append JSONL records to a local file and return the row count."""
    count = 0
    with path.open("a", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
    return count


def append_metric(path: Path, row: dict[str, Any]) -> None:
    """Append one row to metrics.csv, creating the header if needed."""
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def latest_blob_name(bucket: Any, prefix: str, suffix: str) -> str | None:
    """Return the latest blob under prefix with the requested suffix."""
    blobs = [blob for blob in bucket.list_blobs(prefix=prefix) if blob.name.endswith(suffix)]
    if not blobs:
        return None
    blobs.sort(key=lambda b: (b.updated or datetime.min.replace(tzinfo=timezone.utc), b.name), reverse=True)
    return blobs[0].name


def find_manifest_blobs(config: Any, bucket: Any, batches: list[str]) -> list[str]:
    """Find GCS manifest files for selected batches."""
    explicit = str(cfg_value(config, "INPUT_MANIFEST_URI", "") or "").strip()
    if explicit:
        parsed_bucket, blob_name = parse_gcs_uri(explicit)
        if parsed_bucket != bucket.name:
            raise ValueError(f"INPUT_MANIFEST_URI bucket {parsed_bucket} does not match {bucket.name}")
        return [blob_name]
    blobs: list[str] = []
    base = normalize_prefix(cfg_value(config, "MANIFESTS_PREFIX", "processed/keyframes_manifests"))
    for batch_id in batches:
        prefix = f"{base}/dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/profile={cfg_value(config, 'PROFILE_VERSION')}/"
        name = latest_blob_name(bucket, prefix, "shot_segments.csv") or latest_blob_name(bucket, prefix, "frames_manifest.jsonl")
        if name is None:
            raise FileNotFoundError(f"No shot_segments.csv or frames_manifest.jsonl found under gs://{bucket.name}/{prefix}")
        blobs.append(name)
    return blobs


def read_manifest_blob(bucket: Any, blob_name: str) -> pd.DataFrame:
    """Read a CSV or JSONL frame manifest from GCS into a DataFrame."""
    text = bucket.blob(blob_name).download_as_text(timeout=900)
    if blob_name.endswith(".csv"):
        return pd.read_csv(StringIO(text))
    rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    return pd.DataFrame(rows)


def normalize_manifest_records(df: pd.DataFrame, config: Any, batch_id: str | None = None) -> list[dict[str, Any]]:
    """Normalize frame manifest columns into the extractor record contract."""
    if df.empty:
        return []
    df = df.copy()
    if "saved" in df.columns:
        df = df[df["saved"].astype(str).str.lower().isin(["true", "1", "yes"])]
    records: list[dict[str, Any]] = []
    for _, row in df.iterrows():
        image_gcs_uri = str(row.get("image_gcs_uri") or row.get("gcs_uri") or row.get("image_uri") or "").strip()
        if not image_gcs_uri:
            continue
        video_id = str(row.get("video_id") or Path(image_gcs_uri).parent.name).strip()
        frame_idx = int(float(row.get("frame_idx", 0) or 0))
        keyframe_id = str(row.get("keyframe_id") or f"{video_id}_F{frame_idx:06d}")
        records.append({
            "dataset_id": str(row.get("dataset_id") or cfg_value(config, "DATASET_ID")),
            "batch_id": str(row.get("batch_id") or batch_id or "").strip(),
            "video_id": video_id,
            "video_name": str(row.get("video_name") or ""),
            "shot_id": str(row.get("shot_id") or ""),
            "shot_start_frame": int(float(row.get("shot_start_frame", 0) or 0)),
            "shot_end_frame": int(float(row.get("shot_end_frame", 0) or 0)),
            "frame_type": str(row.get("frame_type") or ""),
            "frame_idx": frame_idx,
            "frame_sec": float(row.get("frame_sec", row.get("timestamp", 0)) or 0),
            "timestamp_ms": int(float(row.get("frame_sec", 0) or 0) * 1000),
            "keyframe_id": keyframe_id,
            "image_rel_path": str(row.get("image_rel_path") or f"{video_id}/{Path(image_gcs_uri).name}"),
            "image_gcs_uri": image_gcs_uri,
            "image_storage_key": str(row.get("image_storage_key") or parse_gcs_uri(image_gcs_uri)[1]),
            "fps": float(row.get("fps", 0) or 0),
            "profile_version": str(row.get("profile_version") or cfg_value(config, "PROFILE_VERSION")),
        })
    return records


def discover_frame_records(config: Any, bucket: Any, batches: list[str], max_frames: int | None = None) -> list[dict[str, Any]]:
    """Discover and normalize frame records for selected logical batches."""
    all_records: list[dict[str, Any]] = []
    for blob_name in find_manifest_blobs(config, bucket, batches):
        batch_hint = next((part.split("=", 1)[1] for part in blob_name.split("/") if part.startswith("batch=")), None)
        all_records.extend(normalize_manifest_records(read_manifest_blob(bucket, blob_name), config, batch_hint))
    all_records.sort(key=lambda r: (r["batch_id"], r["video_id"], r["frame_idx"], r["keyframe_id"]))
    return all_records[: int(max_frames)] if max_frames is not None else all_records


def local_frame_path(layout: RunLayout, record: dict[str, Any]) -> Path:
    """Return the local scratch path for one frame record."""
    return layout.frames_dir / record["video_id"] / Path(record["image_gcs_uri"]).name


def download_one_frame(record: dict[str, Any], layout: RunLayout, client: Any) -> dict[str, Any]:
    """Download one frame from GCS into local scratch and return an updated record."""
    started = time.perf_counter()
    bucket_name, blob_name = parse_gcs_uri(record["image_gcs_uri"])
    out_path = local_frame_path(layout, record)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if not out_path.exists() or out_path.stat().st_size == 0:
        client.bucket(bucket_name).blob(blob_name).download_to_filename(str(out_path), timeout=900)
    updated = dict(record)
    updated["local_image_path"] = str(out_path)
    updated["download_ms"] = int((time.perf_counter() - started) * 1000)
    return updated


def download_frames(records: list[dict[str, Any]], layout: RunLayout, client: Any, config: Any) -> list[dict[str, Any]]:
    """Download many frames concurrently from GCS."""
    downloaded: list[dict[str, Any]] = []
    workers = max(1, int(cfg_value(config, "DOWNLOAD_WORKERS", 8)))
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="gcs-download") as pool:
        futures = [pool.submit(download_one_frame, record, layout, client) for record in records]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading frames", disable=not cfg_value(config, "USE_TQDM", True)):
            downloaded.append(future.result())
    downloaded.sort(key=lambda r: (r["batch_id"], r["video_id"], r["frame_idx"], r["keyframe_id"]))
    return downloaded


def iter_batches(items: list[Any], batch_size: int) -> Iterable[list[Any]]:
    """Yield fixed-size batches from a list."""
    batch_size = max(1, int(batch_size))
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def base_annotation(record: dict[str, Any], config: Any, kind: str, run_id: str) -> dict[str, Any]:
    """Create the shared frame annotation JSON payload."""
    return {
        "dataset_id": record["dataset_id"], "batch_id": record["batch_id"], "video_id": record["video_id"],
        "keyframe_id": record["keyframe_id"], "frame_id": record["keyframe_id"], "shot_id": record.get("shot_id", ""),
        "frame_idx": record["frame_idx"], "frame_sec": record["frame_sec"], "timestamp_ms": record.get("timestamp_ms", int(float(record.get("frame_sec", 0)) * 1000)),
        "frame_type": record.get("frame_type", ""), "image_gcs_uri": record["image_gcs_uri"], "image_storage_key": record.get("image_storage_key", ""),
        "kind": kind, "caption": None, "ocr_texts": [], "detected_objects": [], "object_counts": {}, "detections": [],
        "text_value": None, "json_value": {}, "confidence": 1.0, "model_version": str(cfg_value(config, "MODEL_VERSION", "unknown")),
        "annotation_version": str(cfg_value(config, "ANNOTATION_VERSION", cfg_value(config, "EXTRACTOR_VERSION", "v1"))), "run_id": run_id, "created_at": utc_now(),
    }


def upload_standard_artifacts(bucket: Any, layout: RunLayout, success: bool) -> None:
    """Upload standard run artifacts and optional _SUCCESS marker to GCS."""
    for path, content_type in [(layout.annotations_path, "application/jsonl"), (layout.errors_path, "application/jsonl"), (layout.metrics_path, "text/csv"), (layout.summary_path, "application/json"), (layout.log_path, "text/plain")]:
        if path.exists():
            upload_file(bucket, path, layout.output_prefix + path.name, content_type)
    if success:
        upload_text(bucket, "", layout.output_prefix + "_SUCCESS", "text/plain")



def output_base_prefix(config: Any, batch_id: str) -> str:
    """Build the extractor output prefix without run_id for resume discovery."""
    return (
        f"{normalize_prefix(cfg_value(config, 'OUTPUT_PREFIX', 'features/extractors'))}/"
        f"dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/"
        f"frame_profile={cfg_value(config, 'PROFILE_VERSION')}/"
        f"extractor={cfg_value(config, 'EXTRACTOR_NAME')}/"
        f"extractor_version={cfg_value(config, 'EXTRACTOR_VERSION')}/"
    )


def find_resume_annotations_blob(config: Any, bucket: Any, batches: list[str]) -> str | None:
    """Find an annotations.jsonl blob to use for resume filtering."""
    explicit = str(cfg_value(config, "RESUME_ANNOTATIONS_URI", "") or "").strip()
    if explicit:
        parsed_bucket, blob_name = parse_gcs_uri(explicit)
        if parsed_bucket != bucket.name:
            raise ValueError(f"RESUME_ANNOTATIONS_URI bucket {parsed_bucket} does not match {bucket.name}")
        return blob_name
    candidates = []
    for batch_id in batches:
        prefix = output_base_prefix(config, batch_id)
        candidates.extend([blob for blob in bucket.list_blobs(prefix=prefix) if blob.name.endswith("annotations.jsonl")])
    if not candidates:
        return None
    candidates.sort(key=lambda b: (b.updated or datetime.min.replace(tzinfo=timezone.utc), b.name), reverse=True)
    return candidates[0].name


def load_processed_keyframes(config: Any, bucket: Any, batches: list[str]) -> set[str]:
    """Load keyframe IDs already present in a previous successful annotations JSONL."""
    if not cfg_value(config, "SKIP_EXISTING", True) or cfg_value(config, "OVERWRITE", False):
        return set()
    blob_name = find_resume_annotations_blob(config, bucket, batches)
    if not blob_name:
        return set()
    text = bucket.blob(blob_name).download_as_text(timeout=900)
    processed: set[str] = set()
    for line in text.splitlines():
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except Exception:
            continue
        if not row.get("error") and row.get("keyframe_id"):
            processed.add(str(row["keyframe_id"]))
    return processed

def dry_run(config: Any, max_frames: int | None = None) -> dict[str, Any]:
    """Discover frame records without downloading images or running models."""
    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    batches = [str(batch).upper() for batch in cfg_value(config, "BATCHES", [])]
    records = discover_frame_records(config, bucket, batches, max_frames=max_frames)
    return {"status": "DRY_RUN_OK", "bucket": bucket.name, "batches": batches, "planned_frames": len(records), "sample_records": records[:5]}


## 3b. Task Model And Extraction Logic

**Note:** Cell này chứa model và logic riêng của extractor. Các hàm đều trả record theo contract JSONL trung gian, chưa ghi trực tiếp vào Supabase.


In [ ]:
def find_vector_run_prefix(config: Any, bucket: Any, batch_id: str) -> str:
    """Find the latest successful vector embedding run prefix for a batch."""
    explicit = str(cfg_value(config, "VECTOR_RUN_PREFIX", "") or "").strip()
    if explicit:
        if explicit.startswith("gs://"):
            b, p = parse_gcs_uri(explicit.rstrip("/") + "/_")
            if b != bucket.name:
                raise ValueError(f"VECTOR_RUN_PREFIX bucket {b} does not match {bucket.name}")
            return p.rsplit("/", 1)[0].rstrip("/") + "/"
        return normalize_prefix(explicit) + "/"
    base = f"{normalize_prefix(cfg_value(config, 'OUTPUT_PREFIX', 'features/extractors'))}/dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/frame_profile={cfg_value(config, 'PROFILE_VERSION')}/extractor=vector_embedding/extractor_version=fe-vector-embedding-v1/"
    success = latest_blob_name(bucket, base, "_SUCCESS")
    if not success:
        raise FileNotFoundError(f"No successful vector embedding run found under gs://{bucket.name}/{base}")
    return success.rsplit("/", 1)[0] + "/"


def download_vector_artifacts(config: Any, bucket: Any, layout: RunLayout, batch_id: str) -> tuple[Path, Path]:
    """Download vector .npy and map-keyframes CSV files for event segmentation."""
    vector_prefix = find_vector_run_prefix(config, bucket, batch_id)
    local_root = layout.run_dir / "vector_input"
    for blob in bucket.list_blobs(prefix=vector_prefix):
        if blob.name.endswith(".npy") or ("/map-keyframes/" in blob.name and blob.name.endswith(".csv")):
            out = local_root / blob.name[len(vector_prefix):]
            out.parent.mkdir(parents=True, exist_ok=True)
            blob.download_to_filename(str(out), timeout=900)
    features_root = local_root / "features"
    model_dirs = sorted([path for path in features_root.iterdir() if path.is_dir()]) if features_root.exists() else []
    if not model_dirs:
        raise FileNotFoundError("Downloaded vector artifacts do not contain features/<model_folder>.")
    map_dir = local_root / "map-keyframes"
    if not map_dir.exists():
        raise FileNotFoundError("Downloaded vector artifacts do not contain map-keyframes/.")
    return model_dirs[0], map_dir


def _read_csv_safe(path: Path) -> pd.DataFrame:
    """Read CSV files that may contain BOM or non-standard quotes."""
    raw = path.read_bytes()
    try:
        text = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = raw.decode("cp1252")
    text = text.replace("“", '"').replace("”", '"')
    df = pd.read_csv(StringIO(text), sep=None, engine="python")
    df.columns = [str(col).strip().strip('\ufeff"“”') for col in df.columns]
    return df


def segment_one_video(video_id: str, embeddings: np.ndarray, key_map: pd.DataFrame, config: Any) -> tuple[np.ndarray, pd.DataFrame, list[dict[str, Any]]]:
    """Segment keyframes into events using time gap and cosine similarity rules."""
    key_map = key_map.copy()
    key_map["n"] = key_map["n"].astype(int)
    key_map["pts_time"] = key_map["pts_time"].astype(float)
    key_map["frame_idx"] = key_map["frame_idx"].astype(int)
    key_map = key_map.iloc[np.argsort(key_map["pts_time"].values)].reset_index(drop=True)
    embeddings = embeddings[key_map["n"].values - 1].astype("float32")
    embeddings = embeddings / np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-12)
    events, vectors, current = [], [], [0]
    def flush(indices: list[int]) -> None:
        idx = np.array(indices, dtype=int)
        vec = embeddings[idx].mean(axis=0)
        vec = vec / max(float(np.linalg.norm(vec)), 1e-12)
        start_row, end_row = key_map.iloc[indices[0]], key_map.iloc[indices[-1]]
        event_index = len(events)
        keyframe_ids = key_map.iloc[idx]["keyframe_id"].astype(str).tolist() if "keyframe_id" in key_map.columns else [f"{video_id}_F{int(frame):06d}" for frame in key_map.iloc[idx]["frame_idx"]]
        events.append({"event_id": f"{video_id}_E{event_index:06d}", "event_embedding_index": event_index, "embedding_index_0": event_index, "video_id": video_id, "start_n": int(start_row["n"]), "end_n": int(end_row["n"]), "start_sec": float(start_row["pts_time"]), "end_sec": float(end_row["pts_time"]), "start_frame": int(start_row["frame_idx"]), "end_frame": int(end_row["frame_idx"]), "representative_keyframe_id": keyframe_ids[0], "keyframe_ids": keyframe_ids, "keyframe_ns": " ".join(map(str, key_map.iloc[idx]["n"].astype(int).tolist())), "n_keyframes": int(len(idx)), "segmentation_version": cfg_value(config, "SEGMENTATION_VERSION", "fe-scene-detection-v1")})
        vectors.append(vec.astype("float32"))
    for i in range(1, len(key_map)):
        prev_time, cur_time, start_time = float(key_map.loc[i - 1, "pts_time"]), float(key_map.loc[i, "pts_time"]), float(key_map.loc[current[0], "pts_time"])
        current_vec = embeddings[current].mean(axis=0)
        current_vec = current_vec / max(float(np.linalg.norm(current_vec)), 1e-12)
        split = cur_time - prev_time > float(cfg_value(config, "MAX_TIME_GAP_SEC", 6.0)) or float(np.dot(current_vec, embeddings[i])) < float(cfg_value(config, "SCENE_SIMILARITY_THRESHOLD", 0.72))
        max_duration = cfg_value(config, "MAX_EVENT_DURATION_SEC", 45.0)
        if max_duration is not None and cur_time - start_time > float(max_duration):
            split = True
        if split:
            flush(current)
            current = [i]
        else:
            current.append(i)
    flush(current)
    return np.stack(vectors).astype("float32"), pd.DataFrame(events), events


def run_scene_extractor(config: Any, batches: list[str], run_kind: str) -> dict[str, Any]:
    """Run event segmentation from vector embedding artifacts and upload outputs."""
    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    layout = make_run_layout(config, batches[0] if len(batches) == 1 else "all", run_kind)
    setup_logging(layout)
    started = time.perf_counter()
    event_dir, map_dir_out = layout.artifacts_dir / "events", layout.artifacts_dir / "map-event"
    event_dir.mkdir(parents=True, exist_ok=True)
    map_dir_out.mkdir(parents=True, exist_ok=True)
    all_events, summaries = [], []
    for batch_id in batches:
        feature_dir, map_dir = download_vector_artifacts(config, bucket, layout, batch_id)
        for feature_path in tqdm(sorted(feature_dir.glob("*.npy")), desc=f"Segment {batch_id}", disable=not cfg_value(config, "USE_TQDM", True)):
            video_id, t0 = feature_path.stem, time.perf_counter()
            map_path = map_dir / f"{video_id}.csv"
            if not map_path.exists():
                append_jsonl(layout.errors_path, [{"video_id": video_id, "error": f"Missing map-keyframes file {map_path}"}])
                continue
            embeddings, key_map = np.load(feature_path).astype("float32"), _read_csv_safe(map_path)
            event_embeddings, event_map, events = segment_one_video(video_id, embeddings, key_map, config)
            np.save(event_dir / f"{video_id}.npy", event_embeddings)
            event_map.to_csv(map_dir_out / f"{video_id}.csv", index=False)
            all_events.extend(events)
            summaries.append({"video_id": video_id, "num_keyframes": int(len(key_map)), "num_events": int(len(event_map)), "embedding_dim": int(event_embeddings.shape[1]), "seconds": round(time.perf_counter() - t0, 3)})
            append_metric(layout.metrics_path, {"run_id": layout.run_id, "video_id": video_id, "keyframes": int(len(key_map)), "events": int(len(event_map)), "seconds": round(time.perf_counter() - t0, 3)})
    append_jsonl(layout.artifacts_dir / "events.jsonl", all_events)
    write_json(layout.artifacts_dir / "event_summary.json", {"run_id": layout.run_id, "num_videos": len(summaries), "num_events": len(all_events), "videos": summaries})
    success = not layout.errors_path.exists() or layout.errors_path.stat().st_size == 0
    summary = {"run_id": layout.run_id, "status": "SUCCESS" if success else "COMPLETED_WITH_ERRORS", "extractor": cfg_value(config, "EXTRACTOR_NAME"), "dataset_id": cfg_value(config, "DATASET_ID"), "batches": batches, "num_videos": len(summaries), "num_events": len(all_events), "duration_seconds": round(time.perf_counter() - started, 3), "output_prefix": f"gs://{bucket.name}/{layout.output_prefix}", "created_at": utc_now()}
    write_json(layout.summary_path, summary)
    if cfg_value(config, "UPLOAD_TO_GCS", True):
        for path in sorted(layout.artifacts_dir.rglob("*")):
            if path.is_file():
                upload_file(bucket, path, layout.output_prefix + path.relative_to(layout.artifacts_dir).as_posix(), "application/octet-stream")
        if layout.log_path.exists():
            upload_file(bucket, layout.log_path, layout.output_prefix + "run.log", "text/plain")
        if success:
            upload_text(bucket, "", layout.output_prefix + "_SUCCESS", "text/plain")
    return summary


def run_demo(config: Any) -> dict[str, Any]:
    """Run scene segmentation for demo batches."""
    return run_scene_extractor(config, [str(b).upper() for b in cfg_value(config, "DEMO_BATCHES", ["L21"])], "demo")


def run_full(config: Any) -> list[dict[str, Any]]:
    """Run scene segmentation for full selected batches with a safety guard."""
    if cfg_value(config, "CONFIRM_FULL_RUN", "") != "RUN_FULL_DATASET":
        print('Skipped full run. Set CONFIRM_FULL_RUN = "RUN_FULL_DATASET" in the parameter cell and rerun it.')
        return []
    return [run_scene_extractor(config, [str(batch).upper()], f"full_{str(batch).lower()}") for batch in cfg_value(config, "BATCHES", [])]


## 4. Dry Run

**Note:** Cell này kiểm tra GCS credential và tìm vector embedding run mới nhất; không segment event.


In [ ]:
client = make_storage_client(cfg)
bucket = client.bucket(resolve_bucket_name(cfg, require=True))
vector_prefixes = {batch: find_vector_run_prefix(cfg, bucket, batch) for batch in cfg.DEMO_BATCHES}
vector_prefixes


## 5. Demo Run

**Note:** Cell này chạy event segmentation cho `DEMO_BATCHES` từ output của notebook vector embedding.


In [ ]:
demo_summary = run_demo(cfg)
demo_summary


## 6. Full Run

**Note:** Cell này được khóa an toàn. Chỉ chạy toàn bộ khi `CONFIRM_FULL_RUN = "RUN_FULL_DATASET"`.


In [ ]:
full_summaries = run_full(cfg)
full_summaries


## 7. Inspect Latest Local Artifacts

**Note:** Cell này liệt kê output event mới nhất trong Kaggle working directory.


In [ ]:
run_root = Path(cfg.RUN_ROOT)
latest = sorted([path for path in run_root.glob("*") if path.is_dir()], key=lambda path: path.stat().st_mtime, reverse=True)[:5]
for path in latest:
    print(path)
    for artifact in ["artifacts/summary.json", "artifacts/event_summary.json", "artifacts/events.jsonl", "artifacts/metrics.csv", "run.log"]:
        candidate = path / artifact
        if candidate.exists():
            print("  ", candidate, candidate.stat().st_size, "bytes")
